In [1]:
# =========================
# CELL 1) CONFIG + IMPORTS
# =========================
import os
import re
import numpy as np
import pandas as pd

BASE_DIR = r"D:\Thesis\Old\Codes\New"
FILE_PATH = os.path.join(BASE_DIR, "combined_daily.xlsx")

OUT_DIR = os.path.join(BASE_DIR, "panels")
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
N_TICKERS = 100
YEARS_BACK = 10

OUT_PARQUET = os.path.join(OUT_DIR, f"panel_{YEARS_BACK}y_{N_TICKERS}tickers.parquet")

In [2]:
# ==========================================
# CELL 2) HELPERS (CLEAN DUPLICATE TICKERS)
# ==========================================
def clean_ticker_columns_wide(df: pd.DataFrame, date_col: str = "Date") -> pd.DataFrame:
    """
    Fixes columns like 'ZTS.1', 'BRK-B.2' by:
    - stripping trailing .<number>
    - combining duplicates (take first non-null across duplicates)
    """
    df = df.copy()

    # Ensure Date exists
    if date_col not in df.columns:
        raise ValueError(f"Expected '{date_col}' column but got columns: {df.columns.tolist()[:10]} ...")

    # Build mapping from original column -> base column
    new_cols = []
    for c in df.columns:
        if c == date_col:
            new_cols.append(c)
        else:
            base = re.sub(r"\.\d+$", "", str(c)).strip()
            new_cols.append(base)

    df.columns = new_cols

    # If duplicates exist after stripping, combine them
    cols = [c for c in df.columns if c != date_col]
    unique_cols = []
    for c in cols:
        if c not in unique_cols:
            unique_cols.append(c)

    combined = df[[date_col]].copy()
    for c in unique_cols:
        sub = df.loc[:, df.columns == c]  # all duplicates for this ticker
        # combine_first across duplicates left->right
        s = sub.iloc[:, 0]
        for j in range(1, sub.shape[1]):
            s = s.combine_first(sub.iloc[:, j])
        combined[c] = s

    return combined

In [3]:
# =========================
# CELL 3) LOAD ALL 4 SHEETS
# =========================
xlsx = pd.ExcelFile(FILE_PATH)
print("Sheets:", xlsx.sheet_names)

prices_wide  = pd.read_excel(FILE_PATH, sheet_name="Prices")
volumes_wide = pd.read_excel(FILE_PATH, sheet_name="Volumes")
ff_df        = pd.read_excel(FILE_PATH, sheet_name="FF")
gw_df        = pd.read_excel(FILE_PATH, sheet_name="GW")

# Parse dates
for df in [prices_wide, volumes_wide, ff_df, gw_df]:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Clean duplicate ticker columns in Prices/Volumes (e.g., ZTS.1)
prices_wide  = clean_ticker_columns_wide(prices_wide,  date_col="Date")
volumes_wide = clean_ticker_columns_wide(volumes_wide, date_col="Date")

print("Prices shape:", prices_wide.shape)
print("Volumes shape:", volumes_wide.shape)
print("FF shape:", ff_df.shape)
print("GW shape:", gw_df.shape)

Sheets: ['Prices', 'Volumes', 'FF', 'GW']


C:\Users\adiba\AppData\Local\Temp\ipykernel_2612\3946791457.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined[c] = s
C:\Users\adiba\AppData\Local\Temp\ipykernel_2612\3946791457.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined[c] = s
C:\Users\adiba\AppData\Local\Temp\ipykernel_2612\3946791457.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) i

Prices shape: (6434, 504)
Volumes shape: (6434, 504)
FF shape: (6434, 11)
GW shape: (6434, 16)


C:\Users\adiba\AppData\Local\Temp\ipykernel_2612\3946791457.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined[c] = s
C:\Users\adiba\AppData\Local\Temp\ipykernel_2612\3946791457.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined[c] = s
C:\Users\adiba\AppData\Local\Temp\ipykernel_2612\3946791457.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) i

In [4]:
# ==========================================
# CELL 4) SELECT LAST 10 YEARS + 100 TICKERS
# ==========================================
# Use the max date in the dataset as the anchor
max_date = prices_wide["Date"].max()
if pd.isna(max_date):
    raise ValueError("max_date is NaT. Check your Date column parsing.")

start_date = (max_date - pd.DateOffset(years=YEARS_BACK)) + pd.Timedelta(days=1)

# Filter all sheets to the same window
prices_wide_10y  = prices_wide.loc[prices_wide["Date"].between(start_date, max_date)].copy()
volumes_wide_10y = volumes_wide.loc[volumes_wide["Date"].between(start_date, max_date)].copy()
ff_10y           = ff_df.loc[ff_df["Date"].between(start_date, max_date)].copy()
gw_10y           = gw_df.loc[gw_df["Date"].between(start_date, max_date)].copy()

# Ticker universe from intersection of Prices and Volumes columns
price_tickers  = [c for c in prices_wide_10y.columns if c != "Date"]
volume_tickers = [c for c in volumes_wide_10y.columns if c != "Date"]
tickers = sorted(set(price_tickers).intersection(set(volume_tickers)))

if len(tickers) < N_TICKERS:
    raise ValueError(f"Only {len(tickers)} tickers available after intersection, but N_TICKERS={N_TICKERS}.")

rng = np.random.default_rng(SEED)
selected = sorted(rng.choice(tickers, size=N_TICKERS, replace=False).tolist())

print("Date window:", start_date.date(), "to", max_date.date())
print("Total available tickers:", len(tickers))
print("Selected tickers (first 10):", selected[:10])

Date window: 2015-07-01 to 2025-06-30
Total available tickers: 503
Selected tickers (first 10): ['ALB', 'AMGN', 'AMT', 'APA', 'APD', 'APO', 'ARE', 'AXP', 'BBY', 'BKR']


In [5]:
# ==========================================
# CELL 5) BUILD PANEL (LONG) + MERGE FACTORS
# ==========================================
# Keep only selected tickers
prices_sel  = prices_wide_10y[["Date"] + selected].copy()
volumes_sel = volumes_wide_10y[["Date"] + selected].copy()

# Wide -> long
prices_long = prices_sel.melt(id_vars="Date", var_name="ticker", value_name="price")
vols_long   = volumes_sel.melt(id_vars="Date", var_name="ticker", value_name="volume")

# Merge price + volume
panel = prices_long.merge(vols_long, on=["Date", "ticker"], how="inner")

# Merge FF + GW by Date (broadcast across tickers)
ff_cols = [c for c in ff_10y.columns if c != "Date"]
gw_cols = [c for c in gw_10y.columns if c != "Date"]

panel = panel.merge(ff_10y[["Date"] + ff_cols], on="Date", how="left")
panel = panel.merge(gw_10y[["Date"] + gw_cols], on="Date", how="left")

# Final formatting: MultiIndex panel
panel = panel.sort_values(["Date", "ticker"]).set_index(["Date", "ticker"])

print("Final panel shape:", panel.shape)
print("Columns:", panel.columns.tolist()[:15], "...")
panel.head(5)

Final panel shape: (251400, 27)
Columns: ['price', 'volume', 'FF_Mkt-RF', 'FF_SMB', 'FF_HML', 'FF_RF', 'FF5_Mkt-RF', 'FF5_SMB', 'FF5_HML', 'FF5_RMW', 'FF5_CMA', 'FF5_RF', 'GW_DGS10', 'GW_TB3MS', 'GW_BAA'] ...


price   volume  FF_Mkt-RF  FF_SMB  FF_HML  FF_RF  \
Date       ticker                                                          
2015-07-01 ALB      47.259567  1871900     0.0061 -0.0076 -0.0004    0.0   
           AMGN    115.769958  2299100     0.0061 -0.0076 -0.0004    0.0   
           AMT      74.178200  1805600     0.0061 -0.0076 -0.0004    0.0   
           APA      43.806488  3080500     0.0061 -0.0076 -0.0004    0.0   
           APD     100.458244  1261311     0.0061 -0.0076 -0.0004    0.0   

                   FF5_Mkt-RF  FF5_SMB  FF5_HML  FF5_RMW  ...  GW_INDPRO  \
Date       ticker                                         ...              
2015-07-01 ALB         0.0061  -0.0075  -0.0004   0.0021  ...   100.4588   
           AMGN        0.0061  -0.0075  -0.0004   0.0021  ...   100.4588   
           AMT         0.0061  -0.0075  -0.0004   0.0021  ...   100.4588   
           APA         0.0061  -0.0075  -0.0004   0.0021  ...   100.4588   
           APD         0.0061  -0.0075  -0.0004   0.0021  ...   100.4588   

                   GW_UNRATE  GW_TERM  GW_DEF  GW_INF_YoY  GW_IP_YoY  \
Date       ticker                                                      
2015-07-01 ALB           5.3     2.33    0.94    0.179572  -2.065367   
           AMGN          5.3     2.33    0.94    0.179572  -2.065367   
           AMT           5.3     2.33    0.94    0.179572  -2.065367   
           APA           5.3     2.33    0.94    0.179572  -2.065367   
           APD           5.3     2.33    0.94    0.179572  -2.065367   

                   GW_T10Y2Y  GW_TEDRATE  GW_M2SL  GW_VIXCLS  
Date       ticker                                             
2015-07-01 ALB          1.71        0.27  12005.6      18.23  
           AMGN         1.71        0.27  12005.6      18.23  
           AMT          1.71        0.27  12005.6      18.23  
           APA          1.71        0.27  12005.6      18.23  
           APD          1.71        0.27  12005.6      18.23  

[5 rows x 27 columns]

In [6]:
# =========================
# CELL 6) SAVE PANEL
# =========================
panel.to_parquet(OUT_PARQUET, engine="pyarrow", index=True)
print("Saved:", OUT_PARQUET)

Saved: D:\Thesis\Old\Codes\New\panels\panel_10y_100tickers.parquet


In [21]:
import pandas as pd

df = pd.read_parquet(r"D:\Thesis\Old\Codes\New\panels\panel_10y_100tickers.parquet")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nHead:")
print(df.head())

Shape: (251400, 27)
Columns: ['price', 'volume', 'FF_Mkt-RF', 'FF_SMB', 'FF_HML', 'FF_RF', 'FF5_Mkt-RF', 'FF5_SMB', 'FF5_HML', 'FF5_RMW', 'FF5_CMA', 'FF5_RF', 'GW_DGS10', 'GW_TB3MS', 'GW_BAA', 'GW_AAA', 'GW_CPIAUCSL', 'GW_INDPRO', 'GW_UNRATE', 'GW_TERM', 'GW_DEF', 'GW_INF_YoY', 'GW_IP_YoY', 'GW_T10Y2Y', 'GW_TEDRATE', 'GW_M2SL', 'GW_VIXCLS']

Head:
                        price   volume  FF_Mkt-RF  FF_SMB  FF_HML  FF_RF  \
Date       ticker                                                          
2015-07-01 ALB      47.259567  1871900     0.0061 -0.0076 -0.0004    0.0   
           AMGN    115.769958  2299100     0.0061 -0.0076 -0.0004    0.0   
           AMT      74.178200  1805600     0.0061 -0.0076 -0.0004    0.0   
           APA      43.806488  3080500     0.0061 -0.0076 -0.0004    0.0   
           APD     100.458244  1261311     0.0061 -0.0076 -0.0004    0.0   

                   FF5_Mkt-RF  FF5_SMB  FF5_HML  FF5_RMW  ...  GW_INDPRO  \
Date       ticker                        

In [24]:
import os
import numpy as np
import pandas as pd

BASE_DIR = r"D:\Thesis\Old\Codes\New"
PANELS_DIR = os.path.join(BASE_DIR, "panels")
OUT_DIR = os.path.join(BASE_DIR, "feature_engineering_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

RAW_PANEL_PATH = os.path.join(PANELS_DIR, "panel_10y_100tickers.parquet")
OUT_PATH = os.path.join(OUT_DIR, "panel_features_targets_10y_100tickers.parquet")

ALPHA = 0.05  # for VaR/CVaR
SEED = 42

panel = pd.read_parquet(RAW_PANEL_PATH)

# ensure MultiIndex
if not isinstance(panel.index, pd.MultiIndex) or panel.index.names != ["Date", "ticker"]:
    if {"Date","ticker"}.issubset(panel.columns):
        panel["Date"] = pd.to_datetime(panel["Date"])
        panel = panel.set_index(["Date","ticker"]).sort_index()
    else:
        raise ValueError("Panel must have index ['Date','ticker'] or columns Date,ticker.")

panel = panel.sort_index()
panel.index = panel.index.set_levels([pd.to_datetime(panel.index.levels[0]), panel.index.levels[1]])

print("RAW panel shape:", panel.shape)
print("RAW panel columns (first 30):", panel.columns.tolist()[:30])

RAW panel shape: (251400, 27)
RAW panel columns (first 30): ['price', 'volume', 'FF_Mkt-RF', 'FF_SMB', 'FF_HML', 'FF_RF', 'FF5_Mkt-RF', 'FF5_SMB', 'FF5_HML', 'FF5_RMW', 'FF5_CMA', 'FF5_RF', 'GW_DGS10', 'GW_TB3MS', 'GW_BAA', 'GW_AAA', 'GW_CPIAUCSL', 'GW_INDPRO', 'GW_UNRATE', 'GW_TERM', 'GW_DEF', 'GW_INF_YoY', 'GW_IP_YoY', 'GW_T10Y2Y', 'GW_TEDRATE', 'GW_M2SL', 'GW_VIXCLS']


In [26]:
def pick_first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

cols = panel.columns.tolist()

PRICE_COL = pick_first_existing(cols, ["price", "close", "Close", "adj_close", "Adj Close", "PX_LAST", "PRC"])
VOL_COL   = pick_first_existing(cols, ["volume", "Volume", "vol", "VOL"])

if PRICE_COL is None:
    raise ValueError(f"Could not find price column. Available columns sample: {cols[:40]}")
if VOL_COL is None:
    print("WARNING: Could not find volume column. Volume-based features will be NaN.")

print("Using PRICE_COL =", PRICE_COL)
print("Using VOL_COL   =", VOL_COL)

def cvar_from_series(x: pd.Series, alpha=ALPHA):
    x = x.dropna()
    if len(x) == 0:
        return np.nan
    var = x.quantile(alpha)
    tail = x[x <= var]
    return tail.mean() if len(tail) > 0 else var

def add_or_warn(df, name, series):
    df[name] = series
    return name

def nan_col(df, name):
    df[name] = np.nan
    print(f"WARNING: '{name}' cannot be computed with available columns. Filled with NaN.")
    return name

Using PRICE_COL = price
Using VOL_COL   = volume


In [29]:
# =====================================================
# CELL 3) BASE DAILY RETURNS + SAFE ROLLING HELPERS
# =====================================================
import numpy as np
import pandas as pd

# Make sure panel is sorted
panel = panel.sort_index()

# ---------------------------------
# 1) Core daily return series
# ---------------------------------
panel["ret_d"] = panel.groupby(level="ticker")[PRICE_COL].pct_change()

panel["log_ret_d"] = (
    panel.groupby(level="ticker")[PRICE_COL]
         .transform(lambda x: np.log(x).diff())
)

# ---------------------------------
# 2) Helper functions
# ---------------------------------
def cvar_from_series(x: pd.Series, alpha=0.05):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    var = x.quantile(alpha)
    tail = x[x <= var]
    return tail.mean() if len(tail) > 0 else var

def rolling_semivariance(ret: pd.Series, window: int):
    neg = ret.where(ret < 0, 0.0)
    return (neg ** 2).rolling(window).mean()

def drawdown_series(px: pd.Series):
    rolling_peak = px.cummax()
    return (px / rolling_peak) - 1.0

def rolling_max_drawdown(px: pd.Series, window: int):
    dd = drawdown_series(px)
    return dd.rolling(window).min()

def rolling_sharpe(ret: pd.Series, window: int):
    mu = ret.rolling(window).mean()
    sd = ret.rolling(window).std()
    return mu / sd.replace(0, np.nan)

# ---------------------------------
# 3) Precompute safe rolling objects
# ---------------------------------
g_price = panel.groupby(level="ticker")[PRICE_COL]
g_ret   = panel.groupby(level="ticker")["ret_d"]

if VOL_COL is not None:
    g_vol = panel.groupby(level="ticker")[VOL_COL]
else:
    g_vol = None

# ---------------------------------
# 4) Price-derived return features
# ---------------------------------
panel["ret_1d"] = panel["ret_d"]

panel["ret_5d"] = g_price.transform(lambda x: x.pct_change(5))
panel["ret_21d"] = g_price.transform(lambda x: x.pct_change(21))

panel["log_ret_1d"] = panel["log_ret_d"]

panel["momentum_1m"] = g_ret.transform(
    lambda x: (1 + x).rolling(21).apply(np.prod, raw=True) - 1
)

panel["momentum_3m"] = g_ret.transform(
    lambda x: (1 + x).rolling(63).apply(np.prod, raw=True) - 1
)

panel["momentum_6m"] = g_ret.transform(
    lambda x: (1 + x).rolling(126).apply(np.prod, raw=True) - 1
)

panel["momentum_12m"] = g_ret.transform(
    lambda x: (1 + x).rolling(252).apply(np.prod, raw=True) - 1
)

panel["volatility_21d"] = g_ret.transform(lambda x: x.rolling(21).std())
panel["volatility_63d"] = g_ret.transform(lambda x: x.rolling(63).std())

panel["drawdown_63d"] = g_price.transform(lambda x: (x / x.rolling(63).max()) - 1)

panel["zscore_price_63d"] = g_price.transform(
    lambda x: (x - x.rolling(63).mean()) / x.rolling(63).std().replace(0, np.nan)
)

panel["rolling_sharpe_63d"] = g_ret.transform(lambda x: rolling_sharpe(x, 63))

# ---------------------------------
# 5) Volume and liquidity features
# ---------------------------------
if VOL_COL is not None:
    panel["volume_21d_avg"] = g_vol.transform(lambda x: x.rolling(21).mean())

    if "shares_outstanding" in panel.columns:
        panel["turnover"] = panel[VOL_COL] / panel["shares_outstanding"].replace(0, np.nan)
    else:
        panel["turnover"] = np.nan
        print("WARNING: shares_outstanding not found -> turnover set to NaN")

    panel["volume_volatility_21d"] = g_vol.transform(
        lambda x: np.log(x.replace(0, np.nan)).rolling(21).std()
    )

    # OBV
    panel["obv_flow"] = panel[VOL_COL] * np.sign(panel["ret_d"].fillna(0))
    panel["obv"] = panel.groupby(level="ticker")["obv_flow"].cumsum()
    panel.drop(columns=["obv_flow"], inplace=True)

    # PVT
    panel["pvt_flow"] = panel[VOL_COL] * panel["ret_d"].fillna(0)
    panel["pvt"] = panel.groupby(level="ticker")["pvt_flow"].cumsum()
    panel.drop(columns=["pvt_flow"], inplace=True)
else:
    panel["volume_21d_avg"] = np.nan
    panel["turnover"] = np.nan
    panel["volume_volatility_21d"] = np.nan
    panel["obv"] = np.nan
    panel["pvt"] = np.nan
    print("WARNING: volume column not found -> volume/liquidity features set to NaN")

# ---------------------------------
# 6) Risk-based features
# ---------------------------------
panel["var_5p"] = g_ret.transform(lambda x: x.rolling(63).quantile(0.05))

panel["cvar_5p"] = g_ret.transform(
    lambda x: x.rolling(63).apply(lambda s: cvar_from_series(pd.Series(s), 0.05), raw=False)
)

panel["semi_variance_63d"] = g_ret.transform(lambda x: rolling_semivariance(x, 63))

panel["max_drawdown_126d"] = g_price.transform(lambda x: rolling_max_drawdown(x, 126))

# ---------------------------------
# 7) Quick check
# ---------------------------------
print("CELL 3 finished.")
print("Shape after CELL 3:", panel.shape)

check_cols = [
    "ret_d", "log_ret_d", "ret_1d", "ret_5d", "ret_21d",
    "momentum_1m", "momentum_3m", "momentum_6m", "momentum_12m",
    "volatility_21d", "volatility_63d", "drawdown_63d",
    "zscore_price_63d", "rolling_sharpe_63d",
    "volume_21d_avg", "turnover", "volume_volatility_21d", "obv", "pvt",
    "var_5p", "cvar_5p", "semi_variance_63d", "max_drawdown_126d"
]

print("Created columns:")
print([c for c in check_cols if c in panel.columns])

CELL 3 finished.
Shape after CELL 3: (251400, 53)
Created columns:
['ret_d', 'log_ret_d', 'ret_1d', 'ret_5d', 'ret_21d', 'momentum_1m', 'momentum_3m', 'momentum_6m', 'momentum_12m', 'volatility_21d', 'volatility_63d', 'drawdown_63d', 'zscore_price_63d', 'rolling_sharpe_63d', 'volume_21d_avg', 'turnover', 'volume_volatility_21d', 'obv', 'pvt', 'var_5p', 'cvar_5p', 'semi_variance_63d', 'max_drawdown_126d']


In [36]:
# =====================================================
# CELL 4) TARGETS (next-month return + next-month CVaR)
# =====================================================
ALPHA = 0.05

# monthly grid from daily panel
tmp = panel[[PRICE_COL, "ret_d"]].reset_index()
tmp["month"] = tmp["Date"].dt.to_period("M")

# month-end price per ticker
px_me = (
    tmp.sort_values(["ticker", "Date"])
       .groupby(["ticker", "month"], as_index=False)
       .tail(1)
       .rename(columns={PRICE_COL: "px_me"})
       .sort_values(["ticker", "month"])
)

# target_return_next1m: next-month simple return at month-end
px_me["target_return_next1m"] = px_me.groupby("ticker")["px_me"].pct_change().shift(-1)

# next-month CVaR(5%) from daily returns inside next month
def month_cvar(s, alpha=ALPHA):
    s = pd.Series(s).dropna()
    if len(s) == 0:
        return np.nan
    var = s.quantile(alpha)
    tail = s[s <= var]
    return tail.mean() if len(tail) > 0 else var

cvar_by_month = (
    tmp.groupby(["ticker", "month"])["ret_d"]
       .apply(lambda s: month_cvar(s, ALPHA))
       .rename("cvar_this_month")
       .reset_index()
       .sort_values(["ticker", "month"])
)

cvar_by_month["target_cvar_next1m"] = cvar_by_month.groupby("ticker")["cvar_this_month"].shift(-1)

targets_m = (
    px_me.merge(cvar_by_month[["ticker", "month", "target_cvar_next1m"]],
                on=["ticker", "month"], how="left")
         [["ticker", "month", "target_return_next1m", "target_cvar_next1m"]]
)

# broadcast monthly targets back to daily rows (same value for all days in the month)
panel_reset = panel.reset_index()
panel_reset["month"] = panel_reset["Date"].dt.to_period("M")
panel_reset = panel_reset.merge(targets_m, on=["ticker", "month"], how="left")

panel = panel_reset.drop(columns=["month"]).set_index(["Date", "ticker"]).sort_index()

print("CELL 4 finished.")
print("Targets created:", [c for c in panel.columns if c.startswith("target_")])
print("Shape after CELL 4:", panel.shape)

CELL 4 finished.
Targets created: ['target_return_next1m_x', 'target_cvar_next1m_x', 'target_return_next1m_y', 'target_cvar_next1m_y', 'target_return_next1m', 'target_cvar_next1m']
Shape after CELL 4: (251400, 57)


In [37]:
# =====================================================
# CELL 5) FACTOR EXPOSURE FEATURES (6-month rolling)
# beta_MKT_6m, beta_SMB_6m, beta_HML_6m, alpha_FF_6m, r2_FF_6m
# =====================================================

def pick_first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

FF_MKT_RF = pick_first_existing(panel.columns, ["Mkt-RF", "FF_Mkt-RF", "FF_MKT_RF", "mktrf"])
FF_SMB    = pick_first_existing(panel.columns, ["SMB", "FF_SMB", "smb"])
FF_HML    = pick_first_existing(panel.columns, ["HML", "FF_HML", "hml"])
FF_RF     = pick_first_existing(panel.columns, ["RF", "FF_RF", "rf"])

print("Detected FF columns:", {"Mkt-RF": FF_MKT_RF, "SMB": FF_SMB, "HML": FF_HML, "RF": FF_RF})

# monthly return series at month-end
tmp = panel[[PRICE_COL]].reset_index()
tmp["month"] = tmp["Date"].dt.to_period("M")
me = (tmp.sort_values(["ticker","Date"])
         .groupby(["ticker","month"], as_index=False)
         .tail(1)
         .rename(columns={PRICE_COL: "px_me"})
         .sort_values(["ticker","month"]))
me["ret_m"] = me.groupby("ticker")["px_me"].pct_change()

if all([FF_MKT_RF, FF_SMB, FF_HML, FF_RF]):
    # get month-end factor values (one row per month)
    fac = panel.reset_index()
    fac["month"] = fac["Date"].dt.to_period("M")
    fac_me = (fac.sort_values("Date")
                 .groupby("month", as_index=False)
                 .tail(1)[["month", FF_MKT_RF, FF_SMB, FF_HML, FF_RF]])

    me = me.merge(fac_me, on="month", how="left")
    me["excess_ret_m"] = me["ret_m"] - me[FF_RF]

    def rolling_ff_regs(g):
        g = g.sort_values("month").copy()
        y = g["excess_ret_m"].to_numpy(dtype=float)
        X = g[[FF_MKT_RF, FF_SMB, FF_HML]].to_numpy(dtype=float)

        alpha = np.full(len(g), np.nan)
        bmkt  = np.full(len(g), np.nan)
        bsmb  = np.full(len(g), np.nan)
        bhml  = np.full(len(g), np.nan)
        r2    = np.full(len(g), np.nan)

        W = 6
        for i in range(W, len(g)):
            yw = y[i-W:i]
            Xw = X[i-W:i]
            mask = np.isfinite(yw) & np.all(np.isfinite(Xw), axis=1)
            yw = yw[mask]
            Xw = Xw[mask]
            if len(yw) < W:
                continue
            X1 = np.column_stack([np.ones(len(yw)), Xw])
            b = np.linalg.lstsq(X1, yw, rcond=None)[0]
            yhat = X1 @ b
            resid = yw - yhat
            ssr = np.sum(resid**2)
            sst = np.sum((yw - yw.mean())**2)
            r2v = 1 - ssr/sst if sst > 0 else np.nan

            alpha[i] = b[0]
            bmkt[i]  = b[1]
            bsmb[i]  = b[2]
            bhml[i]  = b[3]
            r2[i]    = r2v

        out = g[["ticker","month"]].copy()
        out["alpha_FF_6m"] = alpha
        out["beta_MKT_6m"] = bmkt
        out["beta_SMB_6m"] = bsmb
        out["beta_HML_6m"] = bhml
        out["r2_FF_6m"]    = r2
        return out

    reg = me.groupby("ticker", group_keys=False).apply(rolling_ff_regs)

    # broadcast back to daily by month
    p = panel.reset_index()
    p["month"] = p["Date"].dt.to_period("M")
    p = p.merge(reg, on=["ticker","month"], how="left").drop(columns=["month"])
    panel = p.set_index(["Date","ticker"]).sort_index()

else:
    for c in ["beta_MKT_6m","beta_SMB_6m","beta_HML_6m","alpha_FF_6m","r2_FF_6m"]:
        panel[c] = np.nan
    print("WARNING: FF factor columns not found -> factor exposure features set to NaN")

print("CELL 5 finished.")
print("Shape after CELL 5:", panel.shape)

Detected FF columns: {'Mkt-RF': 'FF_Mkt-RF', 'SMB': 'FF_SMB', 'HML': 'FF_HML', 'RF': 'FF_RF'}


C:\Users\adiba\AppData\Local\Temp\ipykernel_2612\4151400466.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  reg = me.groupby("ticker", group_keys=False).apply(rolling_ff_regs)


CELL 5 finished.
Shape after CELL 5: (251400, 62)


In [38]:
# =====================================================
# CELL 6) MACRO & MARKET FEATURES
# dp, tms, dfy, infl, vix
# =====================================================

for nm in ["dp", "tms", "dfy", "infl"]:
    if nm not in panel.columns:
        panel[nm] = np.nan
        print(f"WARNING: {nm} not found -> set to NaN")

if "vix" not in panel.columns:
    # proxy = 21d market volatility of cross-sectional mean return
    mkt_proxy = panel.groupby(level="Date")["ret_d"].mean()
    panel["vix"] = panel.index.get_level_values("Date").map(mkt_proxy).to_numpy()
    panel["vix"] = panel["vix"].rolling(21).std()
    print("INFO: vix not found -> created proxy vix = 21d std(market return proxy)")

print("CELL 6 finished.")
print("Shape after CELL 6:", panel.shape)

INFO: vix not found -> created proxy vix = 21d std(market return proxy)
CELL 6 finished.
Shape after CELL 6: (251400, 67)


In [42]:
# =====================================================
# CELL 7) CROSS-SECTIONAL + STAT/DEPENDENCY FEATURES
# =====================================================
from sklearn.decomposition import PCA

# sector_dummy, market_cap_log need extra data
panel["sector_dummy"] = np.nan
if "shares_outstanding" in panel.columns:
    panel["market_cap_log"] = np.log(panel[PRICE_COL] * panel["shares_outstanding"].replace(0, np.nan))
else:
    panel["market_cap_log"] = np.nan
    print("WARNING: shares_outstanding not found -> market_cap_log set to NaN")

# relative_return_rank (based on ret_21d)
panel["relative_return_rank"] = panel.groupby(level="Date")["ret_21d"].rank(pct=True)

# market return proxy for correlation/dependence
mkt_ret = panel.groupby(level="Date")["ret_d"].mean()
panel["mkt_ret_proxy"] = panel.index.get_level_values("Date").map(mkt_ret).to_numpy()

# rolling_corr_sp500_63d: corr(asset ret, market proxy) over 63d per ticker
panel["rolling_corr_sp500_63d"] = (
    panel.groupby(level="ticker")["ret_d"]
         .transform(lambda x: x.rolling(63).corr(panel.loc[x.index, "mkt_ret_proxy"]))
)

# ----------------------------
# PCA component 1 (SAFE)
# ----------------------------
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd

pca_inputs = ["ret_1d", "volatility_21d", "momentum_1m"]

def pca_pc1_one_date_safe(df):
    # Extract X
    X = df[pca_inputs].to_numpy(dtype=float)

    # If ALL rows are NaN in any column, PCA is impossible that day
    col_all_nan = np.all(~np.isfinite(X), axis=0)
    if col_all_nan.any():
        return pd.Series(np.nan, index=df.index)

    # Fill NaNs with cross-sectional column means (computed ignoring NaNs)
    col_means = np.nanmean(X, axis=0)
    inds = np.where(~np.isfinite(X))
    X[inds] = np.take(col_means, inds[1])

    # If still NaNs (can happen if col_means has NaN), abort
    if not np.isfinite(X).all():
        return pd.Series(np.nan, index=df.index)

    # Standardize cross-section
    X_std = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)

    # If standardization produced NaN/inf (e.g., constant columns), abort
    if not np.isfinite(X_std).all():
        return pd.Series(np.nan, index=df.index)

    # PCA
    pc1 = PCA(n_components=1).fit_transform(X_std).ravel()
    return pd.Series(pc1, index=df.index)

panel["pca_component_1"] = (
    panel.groupby(level="Date", group_keys=False)
         .apply(pca_pc1_one_date_safe)
)

# ----------------------------
# copula_tail_dependence (SAFE)
# ----------------------------
ALPHA = 0.05
WINDOW = 63

def rolling_lower_tail_dependence(asset: pd.Series, market: pd.Series, window=63, alpha=0.05):
    """
    Nonparametric lower-tail dependence estimate:
      lambda_L = P(Ri <= qi, Rm <= qm) / alpha
    computed over a rolling window.

    Returns a Series aligned to asset.index.
    """
    a = asset.to_numpy(dtype=float)
    m = market.to_numpy(dtype=float)

    out = np.full(len(a), np.nan)

    for i in range(window - 1, len(a)):
        aw = a[i - window + 1 : i + 1]
        mw = m[i - window + 1 : i + 1]

        mask = np.isfinite(aw) & np.isfinite(mw)
        aw = aw[mask]
        mw = mw[mask]

        if len(aw) < max(20, window // 3):
            continue

        qa = np.quantile(aw, alpha)
        qm = np.quantile(mw, alpha)

        joint = np.mean((aw <= qa) & (mw <= qm))
        out[i] = joint / alpha if alpha > 0 else np.nan

    return pd.Series(out, index=asset.index)

# compute per ticker; result aligns to (Date,ticker) index
panel["copula_tail_dependence"] = (
    panel.groupby(level="ticker", group_keys=False)
         .apply(lambda g: rolling_lower_tail_dependence(g["ret_d"], g["mkt_ret_proxy"], WINDOW, ALPHA))
)

print("CELL 7 finished.")
print("Shape after CELL 7:", panel.shape)

CELL 7 finished.
Shape after CELL 7: (251400, 74)


In [43]:
# =====================================================
# CELL 8) TRANSFORMED FEATURES + FINAL PANEL + SAVE
# =====================================================

target_cols = ["target_return_next1m", "target_cvar_next1m"]

# base features (the ones from your list)
base_features = [
    # Price-derived
    "ret_1d","ret_5d","ret_21d","log_ret_1d",
    "momentum_1m","momentum_3m","momentum_6m","momentum_12m",
    "volatility_21d","volatility_63d","drawdown_63d","zscore_price_63d","rolling_sharpe_63d",
    # Volume/liquidity
    "volume_21d_avg","turnover","volume_volatility_21d","obv","pvt",
    # Factor exposure
    "beta_MKT_6m","beta_SMB_6m","beta_HML_6m","alpha_FF_6m","r2_FF_6m",
    # Macro/market
    "dp","tms","dfy","infl","vix",
    # Risk-based
    "var_5p","cvar_5p","semi_variance_63d","max_drawdown_126d",
    # Cross-sectional
    "sector_dummy","market_cap_log","relative_return_rank",
    # Statistical/dependency
    "rolling_corr_sp500_63d","pca_component_1","copula_tail_dependence"
]

# keep only those that exist (some might not exist if earlier cells skipped)
base_features = [c for c in base_features if c in panel.columns]

# standardized + lag1
for c in base_features:
    panel[f"standardized_{c}"] = panel.groupby(level="Date")[c].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-12)
    )
    panel[f"lag1_{c}"] = panel.groupby(level="ticker")[c].shift(1)

feature_cols = base_features + [f"standardized_{c}" for c in base_features] + [f"lag1_{c}" for c in base_features]

final_panel = panel[target_cols + feature_cols].copy()
final_panel = final_panel.dropna(subset=target_cols)

print("FINAL PANEL shape:", final_panel.shape)
print("Tickers:", final_panel.index.get_level_values("ticker").nunique())
print("Dates:", final_panel.index.get_level_values("Date").nunique())
print("Columns:", len(final_panel.columns))

final_panel.to_parquet(OUT_PATH, engine="pyarrow", index=True)
print("Saved:", OUT_PATH)

FINAL PANEL shape: (249400, 116)
Tickers: 100
Dates: 2494
Columns: 116
Saved: D:\Thesis\Old\Codes\New\feature_engineering_outputs\panel_features_targets_10y_100tickers.parquet


In [44]:
import pandas as pd

file_path = r"D:\Thesis\Old\Codes\New\feature_engineering_outputs\panel_features_targets_10y_100tickers.parquet"

panel = pd.read_parquet(file_path)

print("Panel shape:", panel.shape)

Panel shape: (249400, 116)


In [45]:
import os
import numpy as np
import pandas as pd

BASE_DIR = r"D:\Thesis\Old\Codes\New"
IN_PATH = os.path.join(BASE_DIR, "feature_engineering_outputs", "panel_features_targets_10y_100tickers.parquet")

OUT_DIR = os.path.join(BASE_DIR, "feature_tests_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_parquet(IN_PATH).sort_index()

print("Loaded:", df.shape)
print("Columns:", df.columns[:20].tolist())

# keep only month-end rows per ticker (Date, ticker index)
tmp = df.reset_index()
tmp["month"] = tmp["Date"].dt.to_period("M")

# last trading day in each month for each ticker
me = (tmp.sort_values(["ticker","Date"])
          .groupby(["ticker","month"], as_index=False)
          .tail(1)
          .drop(columns=["month"])
     )

me["Date"] = pd.to_datetime(me["Date"])
me = me.set_index(["Date","ticker"]).sort_index()

print("Month-end sample:", me.shape)
print("Tickers:", me.index.get_level_values("ticker").nunique(),
      "Months:", me.index.get_level_values("Date").to_period("M").nunique())

Loaded: (249400, 116)
Columns: ['target_return_next1m', 'target_cvar_next1m', 'ret_1d', 'ret_5d', 'ret_21d', 'log_ret_1d', 'momentum_1m', 'momentum_3m', 'momentum_6m', 'momentum_12m', 'volatility_21d', 'volatility_63d', 'drawdown_63d', 'zscore_price_63d', 'rolling_sharpe_63d', 'volume_21d_avg', 'turnover', 'volume_volatility_21d', 'obv', 'pvt']
Month-end sample: (11900, 116)
Tickers: 100 Months: 119


In [46]:
targets = ["target_return_next1m", "target_cvar_next1m"]

# Features = everything except targets
feature_cols = [c for c in me.columns if c not in targets]

# Drop columns that are fully missing
non_all_nan = [c for c in feature_cols if not me[c].isna().all()]
dropped = sorted(set(feature_cols) - set(non_all_nan))
feature_cols = non_all_nan

print("Total features available:", len(feature_cols))
print("Dropped all-NaN:", len(dropped))
if dropped[:20]:
    print("Examples dropped:", dropped[:20])

# optional: clip extreme values to stabilize MI/RF
def winsorize_series(s, p=0.01):
    lo, hi = s.quantile(p), s.quantile(1-p)
    return s.clip(lo, hi)

# We'll winsorize per feature pooled
me_w = me.copy()
for c in feature_cols:
    me_w[c] = winsorize_series(me_w[c], 0.01)

Total features available: 93
Dropped all-NaN: 21
Examples dropped: ['dfy', 'dp', 'infl', 'lag1_dfy', 'lag1_dp', 'lag1_infl', 'lag1_market_cap_log', 'lag1_sector_dummy', 'lag1_tms', 'lag1_turnover', 'market_cap_log', 'sector_dummy', 'standardized_dfy', 'standardized_dp', 'standardized_infl', 'standardized_market_cap_log', 'standardized_sector_dummy', 'standardized_tms', 'standardized_turnover', 'tms']


In [47]:
from scipy.stats import spearmanr

def monthly_ic_spearman(data: pd.DataFrame, feature: str, target: str):
    # group by month of Date
    d = data[[feature, target]].dropna()
    if d.empty:
        return np.nan, np.nan, np.nan

    # month key
    months = d.index.get_level_values("Date").to_period("M")
    d = d.copy()
    d["month"] = months.values

    ics = []
    for m, g in d.groupby("month"):
        if g[feature].nunique() < 3 or g[target].nunique() < 3:
            continue
        ic = spearmanr(g[feature], g[target]).correlation
        if np.isfinite(ic):
            ics.append(ic)

    if len(ics) < 5:
        return np.nan, np.nan, np.nan

    ic_mean = float(np.mean(ics))
    ic_std  = float(np.std(ics, ddof=1))
    icir    = ic_mean / ic_std if ic_std > 0 else np.nan
    return ic_mean, ic_std, icir


def compute_ic_table(data, features, target):
    rows = []
    for f in features:
        ic_mean, ic_std, icir = monthly_ic_spearman(data, f, target)
        rows.append((f, ic_mean, ic_std, icir))
    out = pd.DataFrame(rows, columns=["feature","IC_mean","IC_std","ICIR"])
    return out

ic_return = compute_ic_table(me_w, feature_cols, "target_return_next1m")
ic_risk   = compute_ic_table(me_w, feature_cols, "target_cvar_next1m")

print("IC computed.")
print("Return IC head:\n", ic_return.sort_values("ICIR", ascending=False).head(10))

IC computed.
Return IC head:
                                feature   IC_mean    IC_std      ICIR
14               volume_volatility_21d  0.027946  0.123799  0.225737
45  standardized_volume_volatility_21d  0.027946  0.123815  0.225707
76          lag1_volume_volatility_21d  0.022851  0.126584  0.180520
9                       volatility_63d  0.041539  0.253353  0.163958
40         standardized_volatility_63d  0.041609  0.253805  0.163942
70                 lag1_volatility_21d  0.036014  0.226588  0.158938
39         standardized_volatility_21d  0.035919  0.227222  0.158081
8                       volatility_21d  0.035292  0.226311  0.155943
56      standardized_semi_variance_63d  0.037235  0.244925  0.152025
71                 lag1_volatility_63d  0.038516  0.255056  0.151009


In [48]:
from sklearn.feature_selection import mutual_info_regression

def compute_mi(data: pd.DataFrame, features, target):
    d = data[features + [target]].dropna()
    if d.empty:
        return pd.DataFrame({"feature": features, "MI": np.nan})

    X = d[features].to_numpy(dtype=float)
    y = d[target].to_numpy(dtype=float)

    mi = mutual_info_regression(X, y, random_state=42)
    out = pd.DataFrame({"feature": features, "MI": mi})
    return out

mi_return = compute_mi(me_w, feature_cols, "target_return_next1m")
mi_risk   = compute_mi(me_w, feature_cols, "target_cvar_next1m")

print("MI computed.")
print(mi_return.sort_values("MI", ascending=False).head(10))

MI computed.
                                feature        MI
61  standardized_copula_tail_dependence  0.075678
86                         lag1_cvar_5p  0.060562
23                               var_5p  0.059343
24                              cvar_5p  0.057677
88               lag1_max_drawdown_126d  0.056347
70                  lag1_volatility_21d  0.052199
26                    max_drawdown_126d  0.051764
85                          lag1_var_5p  0.050383
9                        volatility_63d  0.049649
25                    semi_variance_63d  0.049627


In [49]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

def month_blocks(index_dates):
    return pd.Index(pd.to_datetime(index_dates)).to_period("M")

def time_splits_by_month(dates, n_splits=6, min_train_months=24, test_months=6):
    months = pd.Index(pd.to_datetime(dates)).to_period("M")
    uniq = months.unique().sort_values()

    splits = []
    for k in range(n_splits):
        train_end = min_train_months + k*test_months
        test_end  = train_end + test_months
        if test_end > len(uniq):
            break
        train_months = uniq[:train_end]
        test_months_ = uniq[train_end:test_end]
        splits.append((train_months, test_months_))
    return splits

def rf_perm_importance(data, features, target, n_splits=6):
    d = data[features + [target]].dropna()
    if d.empty:
        return None

    dates = d.index.get_level_values("Date")
    splits = time_splits_by_month(dates, n_splits=n_splits)

    rf_imp = np.zeros(len(features))
    perm_imp = np.zeros(len(features))
    counts = 0

    for train_months, test_months in splits:
        months = pd.Index(pd.to_datetime(d.index.get_level_values("Date"))).to_period("M")
        train_mask = months.isin(train_months)
        test_mask  = months.isin(test_months)

        tr = d.loc[train_mask]
        te = d.loc[test_mask]

        if len(tr) < 1000 or len(te) < 200:
            continue

        Xtr, ytr = tr[features].to_numpy(float), tr[target].to_numpy(float)
        Xte, yte = te[features].to_numpy(float), te[target].to_numpy(float)

        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=5,
            n_jobs=-1,
            random_state=42
        )
        model.fit(Xtr, ytr)

        rf_imp += model.feature_importances_

        perm = permutation_importance(
            model, Xte, yte,
            n_repeats=5,
            random_state=42,
            n_jobs=-1
        )
        perm_imp += perm.importances_mean
        counts += 1

    rf_imp = rf_imp / max(counts, 1)
    perm_imp = perm_imp / max(counts, 1)

    out = pd.DataFrame({
        "feature": features,
        "RF_importance": rf_imp,
        "Perm_importance": perm_imp
    })
    return out

rfp_return = rf_perm_importance(me_w, feature_cols, "target_return_next1m", n_splits=6)
rfp_risk   = rf_perm_importance(me_w, feature_cols, "target_cvar_next1m", n_splits=6)

print("RF+Permutation computed.")
print(rfp_return.sort_values("Perm_importance", ascending=False).head(10))

RF+Permutation computed.
                        feature  RF_importance  Perm_importance
39  standardized_volatility_21d       0.014752         0.012557
32          standardized_ret_5d       0.016104         0.006816
10                 drawdown_63d       0.016721         0.003411
67             lag1_momentum_3m       0.011100         0.003058
7                  momentum_12m       0.014667         0.002649
9                volatility_63d       0.009006         0.002562
49     standardized_beta_SMB_6m       0.017424         0.002065
40  standardized_volatility_63d       0.012556         0.001887
3                    log_ret_1d       0.009626         0.001732
80             lag1_beta_SMB_6m       0.014346         0.001635


In [50]:
def zscore(s):
    s = s.astype(float)
    mu = np.nanmean(s)
    sd = np.nanstd(s)
    return (s - mu) / (sd + 1e-12)

def build_ranking(ic_df, mi_df, rfp_df, out_csv):
    dfm = ic_df.merge(mi_df, on="feature", how="left").merge(rfp_df, on="feature", how="left")

    # z-scores (higher = better). For ICIR, MI, RF, Perm all higher is better.
    dfm["z_ICIR"] = zscore(dfm["ICIR"])
    dfm["z_MI"]   = zscore(dfm["MI"])
    dfm["z_RF"]   = zscore(dfm["RF_importance"])
    dfm["z_PERM"] = zscore(dfm["Perm_importance"])

    # universal score
    dfm["score"] = dfm[["z_ICIR","z_MI","z_RF","z_PERM"]].mean(axis=1)

    dfm = dfm.sort_values("score", ascending=False)
    dfm.to_csv(out_csv, index=False)
    return dfm

rank_return = build_ranking(ic_return, mi_return, rfp_return, os.path.join(OUT_DIR, "ranking_return.csv"))
rank_risk   = build_ranking(ic_risk,   mi_risk,   rfp_risk,   os.path.join(OUT_DIR, "ranking_risk.csv"))

print("Saved rankings to:", OUT_DIR)
print("Top 15 RETURN:\n", rank_return.head(15)[["feature","score","ICIR","MI","RF_importance","Perm_importance"]])
print("Top 15 RISK:\n",   rank_risk.head(15)[["feature","score","ICIR","MI","RF_importance","Perm_importance"]])

Saved rankings to: D:\Thesis\Old\Codes\New\feature_tests_outputs
Top 15 RETURN:
                                 feature     score      ICIR        MI  \
39          standardized_volatility_21d  1.749433  0.158081  0.031189   
70                  lag1_volatility_21d  1.303180  0.158938  0.052199   
40          standardized_volatility_63d  0.995274  0.163942  0.042998   
9                        volatility_63d  0.961166  0.163958  0.049649   
61  standardized_copula_tail_dependence  0.809193 -0.092255  0.075678   
7                          momentum_12m  0.786737  0.047386  0.035525   
8                        volatility_21d  0.770880  0.155943  0.049419   
25                    semi_variance_63d  0.656272  0.150453  0.049627   
32                  standardized_ret_5d  0.653465 -0.025604  0.009733   
71                  lag1_volatility_63d  0.637846  0.151009  0.041251   
69                    lag1_momentum_12m  0.586356  0.068710  0.032863   
59  standardized_rolling_corr_sp500_63d  0.

In [51]:
def corr_prune(data, features, threshold=0.9):
    # Use pooled correlation (fast). For stricter, you can do per-month.
    X = data[features].copy()
    corr = X.corr().abs()

    keep = []
    for f in features:
        if len(keep) == 0:
            keep.append(f)
            continue
        # if highly correlated with any kept feature, skip
        if (corr.loc[f, keep] >= threshold).any():
            continue
        keep.append(f)
    return keep

TOP_N = 60
top_ret = rank_return["feature"].head(TOP_N).tolist()
top_rsk = rank_risk["feature"].head(TOP_N).tolist()

keep_ret = corr_prune(me_w, top_ret, threshold=0.90)
keep_rsk = corr_prune(me_w, top_rsk, threshold=0.90)

print("Return: Top", TOP_N, "-> after corr prune:", len(keep_ret))
print("Risk:   Top", TOP_N, "-> after corr prune:", len(keep_rsk))

pd.Series(keep_ret).to_csv(os.path.join(OUT_DIR, "top_return_corrpruned.csv"), index=False, header=False)
pd.Series(keep_rsk).to_csv(os.path.join(OUT_DIR, "top_risk_corrpruned.csv"), index=False, header=False)

Return: Top 60 -> after corr prune: 35
Risk:   Top 60 -> after corr prune: 36


In [52]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

def topk_curve_ridge(data, ranking_df, target, k_list=(5,10,15,20,30,40,60,80), n_splits=6):
    d_all = data.dropna(subset=[target]).copy()
    dates = d_all.index.get_level_values("Date")
    splits = time_splits_by_month(dates, n_splits=n_splits)

    results = []

    for K in k_list:
        feats = ranking_df["feature"].head(K).tolist()
        d = d_all[feats + [target]].dropna()
        if d.empty:
            results.append((K, np.nan, np.nan))
            continue

        mse_list, r2_list = [], []
        months = pd.Index(pd.to_datetime(d.index.get_level_values("Date"))).to_period("M")

        for train_months, test_months in splits:
            train_mask = months.isin(train_months)
            test_mask  = months.isin(test_months)

            tr = d.loc[train_mask]
            te = d.loc[test_mask]

            if len(tr) < 1000 or len(te) < 200:
                continue

            Xtr, ytr = tr[feats].to_numpy(float), tr[target].to_numpy(float)
            Xte, yte = te[feats].to_numpy(float), te[target].to_numpy(float)

            model = Ridge(alpha=5.0, random_state=42)
            model.fit(Xtr, ytr)
            pred = model.predict(Xte)

            mse_list.append(mean_squared_error(yte, pred))
            r2_list.append(r2_score(yte, pred))

        results.append((K, float(np.mean(mse_list)) if mse_list else np.nan,
                           float(np.mean(r2_list)) if r2_list else np.nan))

    return pd.DataFrame(results, columns=["K","MSE","R2"])

k_list = (5,10,15,20,30,40,60,80)

curve_return = topk_curve_ridge(me_w, rank_return, "target_return_next1m", k_list=k_list, n_splits=6)
curve_risk   = topk_curve_ridge(me_w, rank_risk,   "target_cvar_next1m",  k_list=k_list, n_splits=6)

curve_return.to_csv(os.path.join(OUT_DIR, "topk_curve_return.csv"), index=False)
curve_risk.to_csv(os.path.join(OUT_DIR, "topk_curve_risk.csv"), index=False)

print("Top-K curve (RETURN):\n", curve_return)
print("Top-K curve (RISK):\n", curve_risk)

c:\Users\adiba\anaconda3\envs\Thesis\lib\site-packages\sklearn\linear_model\_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=9.96006e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\adiba\anaconda3\envs\Thesis\lib\site-packages\sklearn\linear_model\_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=6.31381e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\adiba\anaconda3\envs\Thesis\lib\site-packages\sklearn\linear_model\_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=5.24334e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\adiba\anaconda3\envs\Thesis\lib\site-packages\sklearn\linear_model\_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=4.16891e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\adiba\anaconda3\env

Top-K curve (RETURN):
     K       MSE        R2
0   5  0.010114 -0.017851
1  10  0.010148 -0.025464
2  15  0.010579 -0.023553
3  20  0.010538 -0.025003
4  30  0.010238 -0.013935
5  40  0.010238 -0.012001
6  60  0.010327 -0.021112
7  80  0.010487 -0.035822
Top-K curve (RISK):
     K       MSE        R2
0   5  0.000806  0.073895
1  10  0.000788  0.087433
2  15  0.000764  0.094789
3  20  0.000757  0.094592
4  30  0.000760  0.086927
5  40  0.000793 -0.033951
6  60  0.000786 -0.040866
7  80  0.000785 -0.045306
